# PLINK2 DSUB SETUP 

This script first downloads the latest version of PLINK 2 for Linux onto the All of Us Research Program Jupyter platform before uploading it to GCP Workspace Bucket. The uploaded version can then be copied into the working environment of a dsub command. By doing so, the latest release (PLINK v2.0.0-a.7LM, 64-bit Intel, 1 Sep 2025) becomes available for downstream analysis.

To update PLINK 2 to a newer release, replace the version being downloaded and repeat the same steps. This method is used instead of running a Docker image, as directly copying PLINK 2 from GCP Workspace Bucket to the working environment is less computationally expensive and more cost-effective.

The second half of the script verifies the new PLINK 2 module by checking the version number to confirm that the correct release has been downloaded and is functioning properly.

In [ ]:
# Load packages 
import os
import pandas as pd 

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

## DOWNLOAD PLINK2 AND UPLOAD TO GCP 

In [ ]:
%%bash 
# Remove any pre-existing plink2 zip 
rm -rf plink2_linux_x86_64_20250901.zip
# Download the up-to-date module of plink2 for linux system 
wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_20250901.zip .
# Remove pre-exisitng plink2 files 
rm -rf new_plink2
# Create directory for depositing plink2 files 
mkdir new_plink2
# Unzip new plink2 files and deposit them into the "new_plink2" directory 
unzip plink2_linux_x86_64_20250901.zip -d new_plink2 

In [ ]:
# Save new plink2 module to the a directory of in gcp 
!gsutil cp new_plink2/plink2 {my_bucket}/data/plink2_module/

In [ ]:
# Check the successful upload of the new plink2 module 
!gsutil ls {my_bucket}/data/plink2_module/

## TEST RUN PLINK2 MODULE 

This section of the script tests the new PLINK2 module and displays the output of the --help flag.

In [ ]:
%%writefile plink_help.sh
#!/bin/bash

# Exit on error
set -o errexit
# Error on unset vars
set -o nounset

# Copy new plink module into the plink2 
cp "${input_file}" plink2

# Grant permision to run plink2 in environment
chmod +x plink2

# Run plink2 and print out version number 
./plink2 --help >  "${OUTPUT_FILE}"

In [ ]:
# Copy plink_help.sh script to my_bucket 
!gsutil cp plink_help.sh {my_bucket}/data/scripts/

In [ ]:
# Copy plink_help.sh script to my_bucket 
!gsutil cat {my_bucket}/data/scripts/plink_help.sh

In [ ]:
%%bash --out test_ID

# Submit test of plink2 in envirnoment 
# The following script should produce the help page
# of the version of plink2 deposited in the gcp 

# Expected Plink2 Version: 
# PLINK v2.0.0-a.7LM 64-bit Intel (1 Sep 2025) 

# Submit dsub command 
source ~/aou_dsub.bash 

aou_dsub \
  --disk-size 2 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input input_file="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --output OUTPUT_FILE="${WORKSPACE_BUCKET}/data/plink2_module/plink2.txt" \
  --script "${WORKSPACE_BUCKET}/data/scripts/plink_help.sh"
  

In [ ]:
!dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'plink-help--vienose--250921-182534-08' --users 'vienose' --status '*'

In [ ]:
# Check the successful upload of the new plink2 module 
!gsutil ls {my_bucket}/data/plink2_module/

In [ ]:
# Check the successful upload of the new plink2 module 
!gsutil cat {my_bucket}/data/plink2_module/plink2.txt | head